In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')  # NOTA: due punti, non 'memory:'
cursor = conn.cursor()

# DATI DI PRODUZIONE MINERARIA (dati sporchi)
data_produzione = {
    'ID_Mese': ['GEN23', 'feb23', 'MAR23', 'apr23', 'MAG23', 'giu23', 'LUG23', 'ago23', 'SET23', 'ott23', 'NOV23', 'dic23'],
    'Data': ['2023-01-01', '01/02/2023', '2023.03.01', '01-04-2023', '2023/05/01', '2023-06-01', '01/07/2023', '2023.08.01', '01-09-2023', '2023/10/01', '2023-11-01', '31-12-2023'],
    'Sito_estrazione': ['Nord', 'nord', 'SUD', 'Sud', 'Est', 'EST', 'ovest', 'Ovest', 'Nord', 'SUD', 'est', 'Nord'],
    'Minerali_estratti': ['Oro,Argento', 'oro,argento', 'Rame,Oro', 'Oro', 'Argento,Rame', 'Oro,Argento,Rame', 'Rame', 'Oro', 'Argento', 'Oro,Rame', 'Argento', 'Oro,Argento'],
    'Tonni_estratte': ['1250', '1350', '1100', '950', '880', '1020', '980', '1150', '1250', '1080', '1120', '1300'],
    'Oro_grammi': ['2.5', '2.8', '1.8', '2.1', '0', '2.2', '0', '2.4', '0', '2.0', '0', '2.3'],
    'Argento_grammi': ['15', '18', '0', '12', '8.5', '20', '0', '0', '25', '0', '15', '22'],
    'Rame_kg': ['120', '130', '150', '0', '90', '180', '200', '0', '0', '160', '0', '140'],
    'Ore_lavorate': ['720', '700', '680', '650', '620', '690', '640', '710', '680', '660', '670', '700'],
    'Note': ['Produzione regolare', 'Ritardo consegne', 'Manutenzione impianti', 'Nuovo fronte', 'Messa in sicurezza', 'Picco produttivo', 'Guasto macchinari', 'Produzione regolare', 'Sciopero', 'Record estrazione', 'Ferma per inventario', 'Fine anno']
}

# Carica in SQLite
df_produzione = pd.DataFrame(data_produzione)
df_produzione.to_sql('Produzione_Grezza', conn, index=False, if_exists='replace')

print("=== DATI GREZZI PRODUZIONE MINERARIA ===")
print(df_produzione)

In [ ]:
# Query di pulizia
q = """
SELECT 
 -- Pulizia ID_Mese (-> gen23')
UPPER(TRIM(ID_mese)) AS id_mese,

 -- Pulizia Data (-> formato unico YYYY-MM-DD)
CASE
WHEN Data LIKE '__/__/____' THEN 
    SUBSTR(Data, 7, 4) || '-' ||
    SUBSTR(Data, 4, 2) || '-' ||
    SUBSTR(Data, 1, 2)
WHEN Data LIKE '__-__-____' THEN 
    SUBSTR(Data, 7, 4) || '-' ||
    SUBSTR(Data, 4, 2) || '-' ||
    SUBSTR(Data, 1, 2)
WHEN Data LIKE '%.%' THEN 
    REPLACE(Data, '.', '-')
WHEN Data LIKE '%/%' THEN 
    REPLACE(Data, '/', '-')
ELSE Data
END AS data,

 -- Sito_estrazione (-> NORD)
UPPER(TRIM(Sito_estrazione)) AS sito_estrazione,

 -- Tonni_estratte (-> 1250.0)
CAST(TRIM(Tonni_estratte) AS FLOAT) AS ton_estratte,

 -- Oro_grammi (-> 2.5)
CAST(TRIM(Oro_grammi) AS FLOAT) AS oro_g,

 -- Argento_grammi (-> 15.0)
CAST(TRIM(Argento_grammi) AS FLOAT) AS argento_g,

 -- Rame_kg (-> 120.0)
CAST(TRIM(Rame_kg) AS FLOAT) AS rame_Kg,

 -- Ore_lavorate (-> 720.0)
CAST(TRIM(Ore_lavorate) AS FLOAT) AS ore_lavorative,

Note
FROM Produzione_Grezza;
"""

# Creiamo la tabella temporanea con i dati puliti (usa cursor dalla cella precedente)
cursor.execute("CREATE TEMPORARY TABLE Campioni_puliti_temp AS " + q)

# Verifichiamo il contenuto
print("\nProduzione_Grezza --> Puliti!")
df_verifica = pd.read_sql_query("SELECT * FROM Campioni_puliti_temp", conn)
print(df_verifica)

## 📊 Calcolo parametri statistici fondamentali:

### Dopo la pulizia, calcola per Oro_grammi:

Misure di centro:
* Media
* Mediana
* Moda

Misure di spread:
* Range (max - min)
* Varianza
* Deviazione standard
* Coefficiente di variazione (CV = std/media)

Per ogni sito:
* Media Oro per sito
* Media Argento per sito
* Media Rame per sito
* Produttività media (Tonni_estratte / Ore_lavorate)

In [ ]:
q_media_mediana = """
-- Misure di centro
SELECT 
    AVG(oro_g) AS media_oro,
    (SELECT AVG(oro_g) 
     FROM (
         SELECT oro_g
         FROM Campioni_puliti_temp
         WHERE oro_g IS NOT NULL
         ORDER BY oro_g
         LIMIT 2 - (SELECT COUNT(*) FROM Campioni_puliti_temp WHERE oro_g IS NOT NULL) % 2
         OFFSET (SELECT (COUNT(*) - 1) / 2 
                 FROM Campioni_puliti_temp WHERE oro_g IS NOT NULL)
     )) AS mediana_oro
FROM Campioni_puliti_temp;
"""

q_moda = """
-- Moda (valore più frequente)
SELECT 
    oro_g AS moda_oro,
    COUNT(*) AS frequenza
FROM Campioni_puliti_temp
WHERE oro_g IS NOT NULL
GROUP BY oro_g
ORDER BY frequenza DESC
LIMIT 1;
"""

q_spread = """
-- Misure di spread
SELECT 
    MIN(oro_g) AS minimo_oro,
    MAX(oro_g) AS massimo_oro,
    MAX(oro_g) - MIN(oro_g) AS range_oro,
    AVG((oro_g - media_oro) * (oro_g - media_oro)) AS varianza_oro,
    SQRT(AVG((oro_g - media_oro) * (oro_g - media_oro))) AS deviazione_std_oro
FROM Campioni_puliti_temp, 
     (SELECT AVG(oro_g) AS media_oro FROM Campioni_puliti_temp);
"""

df1 = pd.read_sql_query(q_media_mediana, conn)
print(df1)
print("\n")

df2 = pd.read_sql_query(q_moda, conn)
print(df2)
print("\n")

df3 = pd.read_sql_query(q_spread, conn)
print(df3)

### 🎯 "Produzione per sito" significa:
### Analizzare gli indicatori di produzione (quantità di oro, argento, rame, tonnellate, ore lavorate) suddivisi per ciascun sito di estrazione (NORD, SUD, EST, OVEST), per capire:
* Quale sito è più ricco di un determinato minerale
* Quale è più efficiente (tonnellate/ora)
* Quale ha produzione più regolare o più variabile

### Prova a scrivere le query per rispondere a queste domande:
* Quale sito produce più oro in media al mese?
* Quale sito ha la produttività più alta (tonni_estratte / ore_lavorate)?
* Quale sito estrae più minerali in totale (somma di oro+argento+rame)?

In [ ]:
q_produzione = """
SELECT
    ROUND(AVG(oro_g), 2) AS media_oro,
    ROUND(AVG(ton_estratte / ore_lavorative), 2) AS produttivita,  -- Tolto apostrofo, aggiunta virgola
    ROUND(AVG(oro_g + argento_g + (rame_kg * 1000)), 2) AS tot_minerali  -- Convertito kg in g, tolta virgola finale
FROM Campioni_puliti_temp
GROUP BY sito_estrazione;
"""

df4 = pd.read_sql_query(q_produzione, conn)
print(df4)